In [1]:
from steer_core.DataManager import DataManager

from steer_materials.CellMaterials.Base import CurrentCollectorMaterial, InsulationMaterial, SeparatorMaterial
from steer_materials.CellMaterials.Electrode import CathodeMaterial, Binder, ConductiveAdditive, AnodeMaterial

from steer_opencell_design.Components.CurrentCollectors import TablessCurrentCollector
from steer_opencell_design.Formulations.ElectrodeFormulations import CathodeFormulation, AnodeFormulation
from steer_opencell_design.Components.Electrodes import Cathode, Anode
from steer_opencell_design.Components.Separators import Separator
from steer_opencell_design.Constructions.Layups.Laminate import Laminate
from steer_opencell_design.Constructions.ElectrodeAssemblies.JellyRolls import WoundJellyRoll
from steer_opencell_design.AuxillaryComponents.WindingEquipment import RoundMandrel

from steer_opencell_design import __version__

import pandas as pd
from datetime import datetime as dt

In [2]:
db = DataManager()

In [3]:
db.remove_data(table_name='cells', condition="name = 'Vendor G NFM'")

In [4]:
db.get_table_names()

['cells',
 'anode_materials',
 'binder_materials',
 'cathode_materials',
 'conductive_additive_materials',
 'current_collector_materials',
 'insulation_materials',
 'separator_materials']

In [5]:
# Get current collector materials from the database

current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
insulation_material = InsulationMaterial.from_database("Aluminium Oxide, 95%")
separator_material = SeparatorMaterial.from_database("Polypropylene")


In [6]:
# Create the cathode

cathode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=131,
    length=2349,
    coated_width=127,
    insulation_width=3,
    thickness=13
)

cathode_active_material = CathodeMaterial.from_database("NFM111 (Vendor C)")

cathode_formulation = CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=2.76,
    mass_loading=14.48,
    insulation_material=insulation_material,
    insulation_thickness=3
)



In [7]:
# Create the anode

anode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=132,
    length=2427,
    coated_width=127,
    thickness=13
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor A)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.03,
    mass_loading=8.35
)

In [8]:
# make the layup

top_separator = Separator(
    material=separator_material,
    width=130,
    length=2640,
    thickness=12
)

bottom_separator = Separator(
    material=separator_material,
    width=130,
    length=2640,
    thickness=12
)

my_layup = Laminate(
    anode=my_anode,
    cathode=my_cathode,
    top_separator=top_separator,
    bottom_separator=bottom_separator
)



In [9]:
# create the jellyroll assembly

mandrel = RoundMandrel(
    diameter=5,
    length=500,
)

my_jellyroll = WoundJellyRoll(
    laminate=my_layup,
    mandrel=mandrel,
    name="Vendor G NFM",
)

In [10]:
my_jellyroll.roll_properties

,Turns
Anode A Side Coating Turns,42.03
Anode Current Collector Turns,42.03
Anode B Side Coating Turns,42.03
Cathode A Side Coating Turns,39.44
Cathode Current Collector Turns,39.44
Cathode B Side Coating Turns,39.44
Bottom Separator Turns,NaN
Bottom Separator Inner Turns,6.90
Bottom Separator Outer Turns,1.11
Top Separator Turns,NaN


In [11]:
my_jellyroll.get_spiral_plot(height=1300, width=1300).show()

In [12]:

db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_jellyroll.name],
    'object': [my_jellyroll.serialize()],
    'form_factor': ['Cylindrical'],
    'internal_construction': ['Wound'],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))


In [13]:
db.get_data('cells')

,name,object,form_factor,internal_construction,date,version,reference
0,Vendor G NFPP,gASVpzwBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-11 10:52:18,0.4.1,Na/Na+
1,CBAK-32140NS,gASVmwwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-11 15:19:39,0.4.1,Na/Na+
2,Cell 2,gASV5w4BAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-11 15:19:41,0.4.1,Na/Na+
3,Cell 4,gASVDgkBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-11 15:19:44,0.4.1,Na/Na+
4,HiNa-NaCR32140-MP10,gASVmwwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-11 15:19:46,0.4.1,Na/Na+
5,QNAS-S40160NL,gASVmwwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-11 15:19:48,0.4.1,Na/Na+
6,QNAS-S40160RL,gASVmwwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-11 15:19:51,0.4.1,Na/Na+
7,UniGrid-NCO32140,gASVJwsBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-11 15:19:53,0.4.1,Na/Na+
8,Vendor D NFPP,gASVEwABAAAAAACMPnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-11-11 15:19:56,0.4.1,Na/Na+
9,Vendor E NFPP,gASVIQABAAAAAACMPnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-11-11 15:20:00,0.4.1,Na/Na+
